# HDD 08 - Quarantine and replay-buffer readiness

Run all cells in VS Code after Notebook 07. This notebook creates **indices and audits**, not updated neural weights. The next adaptation implementation must use a new experiment ID and compare an offline candidate with the frozen model before any promotion.

Notebook 07 used resampled May date blocks. Repeated dates are not additional history. Candidate eligibility below is a screening rule, not a healthy label or proof of a stable new regime. Failure labels and June remain closed.

The proposal (IT23227804, §3.4.4) requires stable candidate clusters, historical replay, held-out checks and rollback. This notebook implements the data admission part and explicitly records the remaining gates.

In [1]:
from pathlib import Path
import hashlib, json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from IPython.display import display
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
DRIFT_ID='frozen_fleet_drift_v1'
BUFFER_ID='quarantine_replay_readiness_v1'
UP=COMPONENT_ROOT/'results'/RUN_ID/DRIFT_ID
OUT=COMPONENT_ROOT/'results'/RUN_ID/BUFFER_ID
OUT.mkdir(parents=True,exist_ok=True)
def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()
INPUTS={}
def track(p):
    assert p.is_file(), f'Missing upstream file: {p}'
    INPUTS[str(p.relative_to(COMPONENT_ROOT))]=sha256(p)
    return p
record=json.loads(track(UP/'drift_monitoring_complete.json').read_text())
assert record['analysis_complete'] and not record['sealed_test_opened']
assert not record['real_failure_labels_used'] and record['automatic_model_updates']==0
for rel,digest in record['input_hashes'].items():
    assert sha256(COMPONENT_ROOT/rel)==digest, f'Upstream data changed: {rel}'
protocol=record['settings']
SETTINGS={'minimum_panel_fraction':.8,'E_fraction_limit':.10,
          'sector_fraction_limit':.05,'hold_batches':10,
          'candidate_maturation_batches':10,'minimum_unique_dates':14,
          'minimum_candidate_drives':30,'historical_anchor_cap_per_drive':8,
          'historical_anchor_spacing_days':10,'seed':42,
          'upstream_complete_sha256':sha256(UP/'drift_monitoring_complete.json')}
p=OUT/'settings.json'
if p.exists(): assert json.loads(p.read_text())==SETTINGS, 'Changed settings: use a new BUFFER_ID.'
p.write_text(json.dumps(SETTINGS,indent=2))
print('Verified Notebook 07. No model update is enabled.')

Verified Notebook 07. No model update is enabled.


## 1. Reproduce quarantine and admit candidates causally

At an unblocked drift alarm, start a new candidate epoch and clear the previous candidate set. Admit subsequent batches only after ten batches have elapsed. Coverage, temperature-alert and sector-increase guards block admission; any guard also clears the current candidate set. Repeated source dates count once within an epoch. The same date can occur under different synthetic conditions, so manifests are kept separately for each replay.

A candidate epoch requires 14 distinct admitted source dates and 30 distinct drives. These are declared development settings, not tuned or validated acceptance limits. Temporal stability and a held-out acceptance test remain separate required gates.

In [2]:
def as_bool(series):
    if series.dtype==bool: return series
    values=series.astype(str).str.lower()
    assert values.isin(['true','false']).all(), 'Unexpected boolean representation.'
    return values.eq('true')

def admission(table, alarms):
    seen={}; history=[]; candidates=[]; until=-1; epoch=0; alarm_step=None
    for step,row in table.reset_index(drop=True).iterrows():
        reasons=[]
        if row.panel_fraction<SETTINGS['minimum_panel_fraction']: reasons.append('coverage')
        if row.E_alert_fraction>=SETTINGS['E_fraction_limit']: reasons.append('E_alerts')
        if row.sector_increase_fraction>=SETTINGS['sector_fraction_limit']: reasons.append('sector_proxy')
        if reasons:
            until=max(until,step+SETTINGS['hold_batches']); seen={}
        blocked=step<=until
        if step in alarms and not blocked:
            epoch+=1;alarm_step=step;seen={}
        mature=alarm_step is not None and step-alarm_step>=SETTINGS['candidate_maturation_batches']
        key=str(pd.Timestamp(row.source_date).date())
        admitted=bool(not blocked and mature and key not in seen)
        if admitted:
            seen[key]=step
            candidates.append({'epoch':epoch,'source_date':key,'first_admission_step':step})
        history.append({'step':step,'source_date':key,'epoch':epoch,'quarantine_active':blocked,
            'guard_reasons':';'.join(reasons),'alarm':step in alarms,'admitted':admitted,
            'active_unique_dates':len(seen),'active_manifest_dates':';'.join(sorted(seen)),
            'mature':mature})
    return pd.DataFrame(history),pd.DataFrame(candidates,columns=['epoch','source_date','first_admission_step'])

traces=pd.read_csv(track(UP/'all_replay_traces.csv'))
for col in ['drift_detected','quarantine_active','updated','model_update_performed']:
    traces[col]=as_bool(traces[col])
assert not traces.model_update_performed.any()
META=pd.read_parquet(track(COMPONENT_ROOT/'data/features'/RUN_ID/'development/metadata.parquet'))
META['end_date']=pd.to_datetime(META['end_date']).dt.strftime('%Y-%m-%d')
assert not META.duplicated(['serial_number','end_date']).any()
summary=[];AUDITS={};MANIFESTS={}
for (scenario,seed),group in traces.groupby(['scenario','seed'],sort=True):
    seed=int(seed)
    table=pd.read_csv(track(UP/f'replay_inputs_{scenario}_seed{seed}.csv'))
    assert len(table)==protocol['replay_observations']
    alarms=set(group.loc[group.drift_detected,'step'].astype(int))
    audit,admissions=admission(table,alarms)
    # Verify exact reproduction of the Notebook 07 quarantine decision.
    expected=group.groupby('step').quarantine_active.agg(['min','max'])
    assert expected['min'].eq(expected['max']).all()
    np.testing.assert_array_equal(audit.quarantine_active,expected['min'])
    assert not (audit.admitted & audit.quarantine_active).any()
    assert not admissions.duplicated(['epoch','source_date']).any()
    AUDITS[(scenario,seed)]=audit
    # Materialize only the currently active set, not cleared epochs or guarded batches.
    active=audit.iloc[-1].active_manifest_dates
    dates=active.split(';') if active else []
    manifest=META.loc[META.end_date.isin(dates),['serial_number','end_date']].copy()
    manifest['source_split']='development'
    manifest['source_row']=META.index[META.end_date.isin(dates)].to_numpy()
    manifest['scenario']=scenario;manifest['seed']=seed
    manifest['synthetic_values_required']=scenario!='unchanged_control'
    manifest['eligible_for_training']=False
    MANIFESTS[(scenario,seed)]=manifest
    tag=f'{scenario}_seed{seed}'
    audit.to_csv(OUT/f'{tag}_admission_trace.csv',index=False)
    admissions.to_csv(OUT/f'{tag}_admission_history.csv',index=False)
    manifest.to_csv(OUT/f'{tag}_candidate_index.csv',index=False)
    enough=len(dates)>=SETTINGS['minimum_unique_dates'] and manifest.serial_number.nunique()>=SETTINGS['minimum_candidate_drives']
    summary.append({'scenario':scenario,'seed':seed,'alarm_batches':len(alarms),
        'quarantined_batches':int(audit.quarantine_active.sum()),'admission_events':int(audit.admitted.sum()),
        'active_unique_dates':len(dates),'active_unique_drives':manifest.serial_number.nunique(),
        'active_unique_windows':len(manifest),'size_screen_passed':bool(enough),
        'stable_new_regime_validated':False,'heldout_acceptance_passed':False,
        'training_authorized_by_evidence':False})
summary=pd.DataFrame(summary)
summary.to_csv(OUT/'buffer_readiness_by_replay.csv',index=False)
display(summary)

,scenario,seed,alarm_batches,quarantined_batches,admission_events,active_unique_dates,active_unique_drives,active_unique_windows,size_screen_passed,stable_new_regime_validated,heldout_acceptance_passed,training_authorized_by_evidence
0,fleet_sector_shift,42,3,500,0,0,0,0,False,False,False,False
1,fleet_sector_shift,43,3,500,0,0,0,0,False,False,False,False
2,fleet_sector_shift,44,3,500,0,0,0,0,False,False,False,False
3,fleet_temperature_plus_2,42,1,0,31,31,150,4642,True,False,False,False
4,fleet_temperature_plus_2,43,1,0,31,31,150,4642,True,False,False,False
5,fleet_temperature_plus_2,44,1,0,31,31,150,4642,True,False,False,False
6,fleet_temperature_plus_5,42,3,0,41,2,150,300,False,False,False,False
7,fleet_temperature_plus_5,43,2,0,34,31,150,4642,True,False,False,False
8,fleet_temperature_plus_5,44,2,0,38,31,150,4642,True,False,False,False
9,isolated_temperature_spike,42,0,11,0,0,0,0,False,False,False,False


**Important:** candidate indices refer to source windows. A temperature-shift replay needs the matching modified raw values, not the original raw window at that index. This notebook deliberately sets `eligible_for_training=False`; it does not export a misleading ready-to-train tensor. The manifests are diagnostic.

## 2. Build immutable historical replay anchors

Select a bounded, chronologically spaced sample from each original training drive. Historical anchors preserve older coverage; they are not certified healthy. They are already training data and cannot serve as an independent model-acceptance set.

In [3]:
train_path=track(COMPONENT_ROOT/'data/features'/RUN_ID/'train/metadata.parquet')
train=pd.read_parquet(train_path).reset_index(drop=True)
train['source_row']=np.arange(len(train))
train['end_date']=pd.to_datetime(train.end_date)
assert not train.duplicated(['serial_number','end_date']).any()
assert set(train.serial_number).isdisjoint(set(META.serial_number))
anchors=[]
for serial,g in train.groupby('serial_number',sort=True):
    last=None;count=0
    for row in g.sort_values('end_date').itertuples():
        if last is None or (row.end_date-last).days>=SETTINGS['historical_anchor_spacing_days']:
            anchors.append({'serial_number':serial,'end_date':row.end_date.strftime('%Y-%m-%d'),
                'source_split':'train','source_row':row.source_row})
            last=row.end_date;count+=1
            if count>=SETTINGS['historical_anchor_cap_per_drive']:break
anchors=pd.DataFrame(anchors)
assert anchors.serial_number.nunique()==train.serial_number.nunique()
assert not anchors.duplicated(['serial_number','end_date']).any()
anchors.to_csv(OUT/'historical_replay_anchor_index.csv',index=False)
print('Historical replay anchors:',len(anchors),'| drives:',anchors.serial_number.nunique())
print('These indices select the original frozen training tensors; they are not a held-out acceptance set.')

Historical replay anchors: 4885 | drives: 700
These indices select the original frozen training tensors; they are not a held-out acceptance set.


## 3. Check admission logic and record the remaining adaptation gates

The checks below verify guard precedence, hold duration, causal maturity, deduplication and prefix consistency. No scenario name or injected-change label is passed into `admission`; scenario names are used only to organize evaluation files.

Before neural fine-tuning, extend development history and reserve fresh drive-separated development acceptance data. This acceptance set must not be June's sealed final test. Validate candidate cluster stability and specify old-regime reconstruction and alert-budget tolerances before comparing updates. Keep an immutable checkpoint; failed candidates must be rejected without overwriting the deployed checkpoint.

In [4]:
# Small logic checks use artificial summaries, not research outcomes.
base=pd.DataFrame({'source_date':pd.date_range('2021-01-01',periods=40),
    'panel_fraction':1.,'E_alert_fraction':0.,'sector_increase_fraction':0.})
a,m=admission(base,{0})
assert not a.loc[:9,'admitted'].any() and a.loc[10,'admitted']
fault=base.copy();fault.loc[12,'sector_increase_fraction']=1
f,_=admission(fault,{0,12})
assert f.loc[12:22,'quarantine_active'].all()
assert not f.loc[12:22,'admitted'].any()
assert f.loc[12,'active_unique_dates']==0
repeat=base.copy();repeat['source_date']='2021-01-01'
r,rm=admission(repeat,{0});assert len(rm)==1
prefix,_=admission(fault.iloc[:20],{0,12})
pd.testing.assert_frame_equal(prefix,f.iloc[:20].reset_index(drop=True))
print('Admission checks passed: guard precedence, hold, maturity, deduplication and prefix causality.')

gates=pd.DataFrame([
    ['quarantine_and_causal_indexing',True,'Implemented and checked here'],
    ['historical_replay_indices',True,'Original train only; not an independent acceptance set'],
    ['new_chronological_development_history',False,'May date-block resampling does not add history'],
    ['stable_new_regime_evidence',False,'Existing stable partition has no established temporal-regime meaning'],
    ['independent_development_acceptance_set',False,'Reserve additional drives/time before update comparison'],
    ['acceptance_tolerances_preregistered',False,'Define reconstruction and alert-budget margins before candidate fitting'],
    ['offline_candidate_training',False,'Requires preceding data and evidence gates'],
    ['candidate_vs_frozen_acceptance',False,'Not yet evaluated'],
    ['checkpoint_rollback_exercised',False,'Must be tested with an actual candidate checkpoint'],
],columns=['gate','passed','evidence_or_next_action'])
gates.to_csv(OUT/'adaptation_gates.csv',index=False)
display(gates)
for relative,digest in INPUTS.items():
    assert sha256(COMPONENT_ROOT/relative)==digest, f'Input changed: {relative}'
outputs={p.name:sha256(p) for p in sorted(OUT.glob('*.csv'))}
completion={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,
    'run_id':RUN_ID,'buffer_id':BUFFER_ID,'settings':SETTINGS,'input_sha256':INPUTS,
    'output_sha256':outputs,'automatic_model_updates':0,'adaptation_validated':False,
    'real_failure_labels_used':False,'sealed_test_opened':False,
    'next_step':'Audit and acquire additional chronological development data; reserve a separate development acceptance set before offline adaptation.',
    'limitations':['candidate screening does not label health','all replay candidates reuse inspected May source dates',
        'synthetic candidate indices need matching transformed values','historical anchors are not held-out data',
        'no neural training, acceptance comparison or checkpoint rollback has been performed']}
(OUT/'replay_buffer_complete.json').write_text(json.dumps(completion,indent=2,allow_nan=False))
print('NOTEBOOK 08 COMPLETE — quarantine and replay manifests audited.')
print('Model updates: 0. Next: chronological data expansion and a development acceptance split.')
print('Results:',OUT)
print('Share buffer_readiness_by_replay.csv, adaptation_gates.csv and this executed notebook.')

Admission checks passed: guard precedence, hold, maturity, deduplication and prefix causality.


,gate,passed,evidence_or_next_action
0,quarantine_and_causal_indexing,True,Implemented and checked here
1,historical_replay_indices,True,Original train only; not an independent accept...
2,new_chronological_development_history,False,May date-block resampling does not add history
3,stable_new_regime_evidence,False,Existing stable partition has no established t...
4,independent_development_acceptance_set,False,Reserve additional drives/time before update c...
5,acceptance_tolerances_preregistered,False,Define reconstruction and alert-budget margins...
6,offline_candidate_training,False,Requires preceding data and evidence gates
7,candidate_vs_frozen_acceptance,False,Not yet evaluated
8,checkpoint_rollback_exercised,False,Must be tested with an actual candidate checkp...


NOTEBOOK 08 COMPLETE — quarantine and replay manifests audited.
Model updates: 0. Next: chronological data expansion and a development acceptance split.
Results: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\results\single_model_q1q2_2021_v1\quarantine_replay_readiness_v1
Share buffer_readiness_by_replay.csv, adaptation_gates.csv and this executed notebook.
